# World Happiness Data Mining Project

**Student:** Begimai Ashyrbekova

This notebook demonstrates a full data preprocessing and data mining workflow using a World Happiness style dataset from 2020 to 2024.


## 1. Import Libraries
The first step is to import the Python libraries needed for data cleaning, visualization, dimensionality reduction, and machine learning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load the Dataset
This dataset was intentionally made a little messy so the preprocessing stage is more realistic.


In [ ]:
df = pd.read_csv('world_happiness_data.csv')
df.head()


## 3. Check Data Quality Problems
Here I inspect the shape, data types, missing values, and duplicated rows.


In [ ]:
print('Shape before cleaning:', df.shape)
print('\nData types before cleaning:\n', df.dtypes)
print('\nMissing values before cleaning:\n', df.isna().sum())
print('\nDuplicate rows before cleaning:', df.duplicated().sum())


## 4. Data Cleaning
In this section I clean inconsistent country names, remove duplicate rows, convert numeric columns to correct format, and fill missing values.


In [ ]:
# Clean text columns
df['Country'] = df['Country'].astype(str).str.strip().str.title().str.replace('  ', ' ', regex=False)

# Convert columns to numeric where needed
numeric_cols = ['GDP_per_Capita','Social_Support','Healthy_Life_Expectancy',
                'Freedom_to_Make_Choices','Generosity','Perceptions_of_Corruption','Happiness_Score']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove duplicates
df = df.drop_duplicates()

# Fill missing values with median
imputer = SimpleImputer(strategy='median')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

print('Shape after cleaning:', df.shape)
print('\nMissing values after cleaning:\n', df.isna().sum())
df.head()


## 5. Basic Exploratory Analysis
I calculate the correlation of the numerical features with happiness score to see which variables are most related.


In [ ]:
corr = df[numeric_cols].corr(numeric_only=True)['Happiness_Score'].sort_values(ascending=False)
corr


In [ ]:
corr.drop('Happiness_Score').plot(kind='bar', figsize=(9,5))
plt.title('Correlation with Happiness Score')
plt.ylabel('Correlation')
plt.tight_layout()
plt.show()


## 6. Data Reduction with PCA
PCA reduces the number of dimensions while preserving as much information as possible.


In [ ]:
X = df[['GDP_per_Capita','Social_Support','Healthy_Life_Expectancy',
        'Freedom_to_Make_Choices','Generosity','Perceptions_of_Corruption']]
y = df['Happiness_Score']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)
explained = pd.DataFrame({
    'Principal Component':[f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
    'Explained Variance Ratio':pca.explained_variance_ratio_
})
explained


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_, marker='o')
plt.title('PCA Explained Variance Ratio')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.tight_layout()
plt.show()


## 7. Attribute Subset Selection
Here I use SelectKBest with f-regression to find the strongest features for predicting happiness score.


In [ ]:
selector = SelectKBest(score_func=f_regression, k=4)
selector.fit(X, y)
selected_features = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector.scores_
}).sort_values(by='Score', ascending=False)
selected_features


## 8. Train-Test Split
To evaluate the models fairly, I split the data into training and testing sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training set:', X_train.shape)
print('Testing set:', X_test.shape)


## 9. Linear Regression
Linear Regression is used here as a baseline model.


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

lr_results = {
    'MAE': mean_absolute_error(y_test, lr_pred),
    'RMSE': mean_squared_error(y_test, lr_pred) ** 0.5,
    'R2': r2_score(y_test, lr_pred)
}
lr_results


## 10. Random Forest Regression
Random Forest can capture more complex relationships between the variables.


In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_results = {
    'MAE': mean_absolute_error(y_test, rf_pred),
    'RMSE': mean_squared_error(y_test, rf_pred) ** 0.5,
    'R2': r2_score(y_test, rf_pred)
}
rf_results


In [ ]:
comparison = pd.DataFrame([lr_results, rf_results], index=['Linear Regression','Random Forest'])
comparison


In [ ]:
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)
importances


In [ ]:
importances.plot(kind='bar', x='Feature', y='Importance', figsize=(9,5), legend=False)
plt.title('Random Forest Feature Importance')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()


## 11. Conclusion
This project showed the full workflow of a data mining task. I started with messy data containing missing values, duplicated rows, mixed data types, and inconsistent text labels. After cleaning the data, I applied PCA for dimensionality reduction and SelectKBest for attribute subset selection. Finally, I trained both Linear Regression and Random Forest models to predict happiness score. Based on the results, the strongest predictors were usually GDP per capita, social support, health, and corruption perception. This project helped demonstrate how preprocessing improves data quality and how machine learning can be used to discover meaningful patterns in happiness data.
